In [1]:

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, Bidirectional, LSTM, Dense, Dropout, LayerNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

import bz2
import csv
import re

In [6]:
import pandas as pd

train_path = r"C:\Users\Lenovo\OneDrive\Documents\Data set\Sentiment_Anlaysis\train.ft.txt.bz2"
test_path = r"C:\Users\Lenovo\OneDrive\Documents\Data set\Sentiment_Anlaysis\test.ft.txt.bz2"

training_data = pd.read_csv(train_path, compression='bz2', header=None, sep='\t')
test_data = pd.read_csv(test_path, compression='bz2', header=None, sep='\t')


In [7]:
training_data = training_data.sample(12000, random_state=42)
test_data = test_data.sample(3000, random_state=42)

In [8]:
training_data[0]

2079998    __label__1 Expensive Junk: This product consis...
1443106    __label__1 Toast too dark: Even on the lowest ...
3463669    __label__2 Excellent imagery...dumbed down sto...
2914699    __label__1 Are we pretending everyone is marri...
1603231    __label__1 Not worth your time: Might as well ...
                                 ...                        
1256389    __label__1 Wow.: For a long awaited album that...
1774982    __label__1 Disappointing: Upon first removing ...
563145     __label__1 Readers digest has changed!: Reader...
3313215    __label__1 see and learn: I was very disappoin...
804344     __label__1 A 15 Year Olds Objections to Fight ...
Name: 0, Length: 12000, dtype: object

In [10]:
# Split the data into labels and texts
training_labels = [int(re.findall(r'__label__(\d)', line)[0]) for line in training_data[0].values]
training_texts = [re.sub(r'__label__\d ', '', line) for line in training_data[0].values]

test_labels = [int(re.findall(r'__label__(\d)', line)[0]) for line in test_data[0].values]
test_texts = [re.sub(r'__label__\d ', '', line) for line in test_data[0].values]

# Convert labels to binary (0 and 1)
training_labels = [0 if label == 1 else 1 for label in training_labels]
test_labels = [0 if label == 1 else 1 for label in test_labels]

In [12]:
print(test_labels[:4])

[1, 0, 1, 1]


# Text Cleaning

In [18]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer



def clean_text(text):
    ps=PorterStemmer()
    stop_words=set(stopwords.words('english'))

    # Convert to lowercase
    text = text.lower()

     # Remove non-alphabetic characters (keep spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenize and remove stopwords, apply stemming
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words]
    # Join words back into a string
    return " ".join(words)

# Apply function to training and test texts
training_texts = [clean_text(text) for text in training_texts]
test_texts = [clean_text(text) for text in test_texts]
    

In [19]:
training_texts[0]

'expens junk product consist piec thin flexibl insul materi adhes back velcro white electr tapeproblem instruct three pictur littl inform velcro crumpl receiv stronger adhes tri disengag velcro piec came paint ceil white electr tape horribl cheap narrow fell less hour price ripoffi build easier use cheaper attract higher rvalu surpris amazon even list junk'

# Tokenization